# Cluster {{cluster_id}} Diagnostic Plots

Photometric and spatial diagnostics for cluster **{{cluster_id}}** and its member galaxies in the DES Y6 WaZP catalog.

**References**
- [Benoist et al. 2025, A&A](https://doi.org/10.1051/0004-6361/202555607) — DES Y6 WaZP catalog
- [Aguena et al. 2021, MNRAS 502, 4435](https://doi.org/10.1093/mnras/stab264) — WaZP method
- [Castignani & Benoist 2016, A&A 595, A111](https://doi.org/10.1051/0004-6361/201528009) — membership probability $P_{\mathrm{mem}}$
- [LIneA WaZP data products](https://data.linea.org.br/en/sci_products/wazp.html)

**Legend:** Red dashed circles mark the central galaxy (CG) when identified. Gray crosshairs indicate the cluster center on spatial plots. Gray dashed circles show the cluster radius ($R_{\mathrm{amin}}$).

In [ ]:
# canvas-variables
import json
main_table_metadata = json.loads("""{{main_table_metadata}}""")
main_record = json.loads("""{{main_record}}""")
related_table_metadata = json.loads("""{{related_table_metadata}}""")
related_table_data = json.loads("""{{related_table_data}}""")

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.collections import CircleCollection
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")

# DES Y6 WaZP uses the z band as the reference for detection and richness (Benoist et al. 2025)
FIG_WIDTH = 14.0  # shared width for all figures (alignment)
CENTER_WIDTH_RATIO = 0.62  # single-panel plots centered in FIG_WIDTH
REF_MAG = "mag_z"
CG_MARKER_AREA = 100.0  # points^2; CircleCollection centers on (x, y)
PMEM_SCATTER_SIZE = 20  # scatter marker area (points^2), colored by Pmem

LBL = {
    "zphot_cluster": r"$z_{\mathrm{phot}}$ (cluster)",
    "zphot_galaxy": r"$z_{\mathrm{phot}}$ (galaxy)",
    "pmem": r"$P_{\mathrm{mem}}$",
    "mag_g": r"$m_g$",
    "mag_r": r"$m_r$",
    "mag_i": r"$m_i$",
    "mag_z": r"$m_z$",
    "mag_y": r"$m_Y$",
    "g_r": r"$g-r$",
    "r_i": r"$r-i$",
    "i_z": r"$i-z$",
    "g-r": r"$g-r$",
    "r-i": r"$r-i$",
    "i-z": r"$i-z$",
    "ra": "RA (deg)",
    "dec": "Dec (deg)",
    "dra": r"$\Delta\mathrm{RA}$ (arcsec)",
    "ddec": r"$\Delta\mathrm{Dec}$ (arcsec)",
    "counts": "Count",
    "sum_pmem": r"$\sum P_{\mathrm{mem}}$",
    "weighted": r"$P_{\mathrm{mem}}$-weighted",
    "galaxies": "Galaxy counts",
    "offsets": "Offsets from center",
    "sky": "Sky position",
    "cg": "CG",
    "wazp_center": "WaZP center",
    "central_galaxy": "Central galaxy",
}

CMD_X_RANGE = (15, 24)
CMD_Y_RANGE = (-0.5, 1.8)

cluster = main_record
members = pd.DataFrame(related_table_data)
cluster_name = cluster.get("name") or cluster.get("id") or cluster.get("meta_id")

In [ ]:
def _get(d, *keys, default=None):
    for key in keys:
        if key in d and d[key] not in (None, ""):
            return d[key]
    return default


def _filled(val):
    return val is not None and str(val).strip() != ""


def _flt(val, default=np.nan):
    if val is None:
        return default
    return float(val)


def _norm_id(val):
    if val is None:
        return None
    s = str(val).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s


def _valid_cg_id(val):
    cg_id = _norm_id(val)
    if cg_id is None:
        return None
    try:
        if int(float(cg_id)) <= 0:
            return None
    except ValueError:
        return None
    return cg_id


def _member_ids(members):
    ids = pd.Series(index=members.index, dtype=object)
    for col in ("id", "meta_id"):
        if col in members.columns:
            ids = ids.fillna(members[col].map(_norm_id))
    return ids


def _bcg_mask(members, cluster):
    mask = pd.Series(False, index=members.index)
    if members.empty:
        return mask

    if "flag_cg" in members.columns:
        fg = members["flag_cg"]
        if fg.dtype == bool:
            mask = mask | fg.fillna(False)
        else:
            s = fg.astype(str).str.strip().str.lower()
            mask = mask | s.isin(["true", "1", "t", "yes"])
            num = pd.to_numeric(fg, errors="coerce")
            mask = mask | (num == 1)

    cg_id = _valid_cg_id(_get(cluster, "id_cg"))
    if cg_id is not None:
        mids = _member_ids(members)
        if mids.notna().any():
            mask = mask | (mids == cg_id)

    cg_ra = _get(cluster, "ra_cg")
    cg_dec = _get(cluster, "dec_cg")
    if _filled(cg_ra) and _filled(cg_dec) and {"ra", "dec"}.issubset(members.columns):
        ra = pd.to_numeric(members["ra"], errors="coerce")
        dec = pd.to_numeric(members["dec"], errors="coerce")
        dra = (ra - float(cg_ra)) * np.cos(np.radians(float(cg_dec))) * 3600
        ddec = (dec - float(cg_dec)) * 3600
        near = dra.pow(2).add(ddec.pow(2)).pow(0.5).lt(2.0)
        mask = mask | near.fillna(False)

    if not mask.any() and _filled(cg_ra) and _filled(cg_dec) and {"ra", "dec"}.issubset(members.columns):
        ra = pd.to_numeric(members["ra"], errors="coerce")
        dec = pd.to_numeric(members["dec"], errors="coerce")
        dra = (ra - float(cg_ra)) * np.cos(np.radians(float(cg_dec))) * 3600
        ddec = (dec - float(cg_dec)) * 3600
        dist2 = dra.pow(2).add(ddec.pow(2))
        if dist2.notna().any():
            idx = dist2.idxmin()
            if dist2.loc[idx] <= 4.0:
                mask.loc[idx] = True

    return mask


def _to_float_array(values):
    return pd.to_numeric(pd.Series(values), errors="coerce").to_numpy(dtype=float)


def _finite_mask(values):
    return np.isfinite(_to_float_array(values))


def _cg_xy_mask(bcg_mask, x, y):
    return bcg_mask & _finite_mask(x) & _finite_mask(y)


def _plot_cg(ax, x, y, label=None):
    xv = _to_float_array(x)
    yv = _to_float_array(y)
    ok = np.isfinite(xv) & np.isfinite(yv)
    if not ok.any():
        return
    pts = np.column_stack([xv[ok], yv[ok]])
    sizes = np.full(len(pts), CG_MARKER_AREA)
    coll = CircleCollection(
        sizes,
        offsets=pts,
        transOffset=ax.transData,
        facecolors="none",
        edgecolors="red",
        linewidths=0.8,
        linestyles="dashed",
        zorder=10,
    )
    if label is not None:
        coll.set_label(label)
    ax.add_collection(coll)


def _cluster_radius_arcsec(cluster):
    val = _flt(_get(cluster, "radius_amin"))
    return val * 60.0 if np.isfinite(val) and val > 0 else np.nan


def _add_offset_radius_circle(ax, radius_arcsec):
    if not np.isfinite(radius_arcsec) or radius_arcsec <= 0:
        return
    circ = plt.Circle(
        (0, 0), radius_arcsec, fill=False,
        edgecolor="gray", linestyle="--", linewidth=0.8, zorder=1,
        label="Cluster radius",
    )
    ax.add_patch(circ)


def _add_sky_radius_circle(ax, ra, dec, radius_deg):
    if not np.isfinite(radius_deg) or radius_deg <= 0:
        return
    circ = plt.Circle(
        (ra, dec), radius_deg, fill=False,
        edgecolor="gray", linestyle="--", linewidth=0.8, zorder=1,
        label="Cluster radius",
    )
    ax.add_patch(circ)


def _symmetric_arcsec_limits(ax, x, y, radius_arcsec, pad=1.12):
    vals = []
    for s in (x, y):
        s = pd.to_numeric(s, errors="coerce").dropna()
        if not s.empty:
            vals.append(s.abs().max())
    if np.isfinite(radius_arcsec):
        vals.append(radius_arcsec)
    if not vals:
        return
    lim = max(max(vals) * pad, 1.0)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)


def _set_equal_sky_axes(ax):
    ax.set_aspect("equal", adjustable="box")
    if hasattr(ax, "set_box_aspect"):
        ax.set_box_aspect(1)


def _finite_xy_mask(x, y):
    return _finite_mask(x) & _finite_mask(y)


def _format_ra_dec_ticks(ax, cluster_ra, cluster_dec):
    cos_dec = np.cos(np.radians(cluster_dec))
    ax.xaxis.set_major_formatter(
        FuncFormatter(lambda x, _: f"{cluster_ra + x / (3600 * cos_dec):.4f}")
    )
    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda y, _: f"{cluster_dec + y / 3600:.4f}")
    )
    ax.set_xlabel(LBL["ra"])
    ax.set_ylabel(LBL["dec"])


def _subplots_grid(nrows=1, ncols=1, height=5.0, **kwargs):
    """Multi-panel figures at the shared notebook width."""
    return plt.subplots(
        nrows, ncols, figsize=(FIG_WIDTH, height), constrained_layout=True, **kwargs,
    )


def _centered_fig(height=5.0, width_ratio=CENTER_WIDTH_RATIO):
    """Single-panel figure centered within the shared width."""
    fig = plt.figure(figsize=(FIG_WIDTH, height), constrained_layout=True)
    pad = (1.0 - width_ratio) / 2.0
    gs = fig.add_gridspec(1, 1, left=pad, right=1.0 - pad)
    ax = fig.add_subplot(gs[0, 0])
    return fig, ax

def choose_cmd_color(z):
    """Recommended color vs. redshift; x-axis = m_z (Y6 reference band)."""
    if z < 0.35:
        return "g_r", REF_MAG, "g-r"
    if z < 0.75:
        return "r_i", REF_MAG, "r-i"
    return "i_z", REF_MAG, "i-z"


z_cluster = _flt(_get(cluster, "zphot"))
cluster_ra = _flt(_get(cluster, "ra", "meta_ra"))
cluster_dec = _flt(_get(cluster, "dec", "meta_dec"))
ngals = _get(cluster, "ngals")
snr = _get(cluster, "snr")
radius_mpc = _get(cluster, "radius_mpc")
act_id = _get(cluster, "sze_act_id")
spt_id = _get(cluster, "sze_spt_id")
in_cosmo = _get(cluster, "in_cosmo")
sze_matched = _filled(act_id) or _filled(spt_id)

if members.empty:
    bcg_mask = pd.Series(dtype=bool)
else:
    mag_cols = [c for c in ("mag_g", "mag_r", "mag_i", "mag_z", "mag_y") if c in members.columns]
    for col in mag_cols + ["pmem", "zphot", "ra", "dec"]:
        if col in members.columns:
            members[col] = pd.to_numeric(members[col], errors="coerce")

    if {"mag_g", "mag_r", "mag_i", "mag_z"}.issubset(members.columns):
        members["g_r"] = members["mag_g"] - members["mag_r"]
        members["r_i"] = members["mag_r"] - members["mag_i"]
        members["i_z"] = members["mag_i"] - members["mag_z"]

    col_y, x_key, color_label = choose_cmd_color(z_cluster)

    bcg_mask = _bcg_mask(members, cluster)

    radius_arcsec = _cluster_radius_arcsec(cluster)
    radius_deg = radius_arcsec / 3600.0 if np.isfinite(radius_arcsec) else np.nan

    if np.isfinite(cluster_ra) and np.isfinite(cluster_dec) and {"ra", "dec"}.issubset(members.columns):
        members["dra_arcsec"] = (members["ra"] - cluster_ra) * np.cos(np.radians(cluster_dec)) * 3600
        members["ddec_arcsec"] = (members["dec"] - cluster_dec) * 3600


## Primary color-magnitude diagram

Two-dimensional histogram of member galaxies in the recommended color–magnitude plane for this cluster's photometric redshift. The left panel shows galaxy counts per bin; the right panel shows the sum of membership probabilities ($\sum P_{\mathrm{mem}}$) per bin. The central galaxy (CG) is marked with a red dashed circle when identified.

Recommended color index as a function of cluster $z_{\mathrm{phot}}$ (Aguena et al. 2021; Benoist et al. 2025):

$$c = \begin{cases} g-r & z_{\mathrm{phot}} < 0.35 \\ r-i & 0.35 \leq z_{\mathrm{phot}} < 0.75 \\ i-z & z_{\mathrm{phot}} \geq 0.75 \end{cases}$$

DES Y6 uses $m_z$ as the reference magnitude (Benoist et al. 2025).

In [ ]:
if not members.empty:
    x = members[x_key]
    y = members[col_y]
    w = members["pmem"] if "pmem" in members.columns else None
    valid = _finite_xy_mask(x, y)
    weight_valid = valid & w.notna() & np.isfinite(w) if w is not None else valid

    fig, axes = _subplots_grid(1, 2, height=5.0)

    if valid.any():
        h0 = axes[0].hist2d(
            x[valid], y[valid], bins=50, range=[CMD_X_RANGE, CMD_Y_RANGE], cmap="viridis",
        )
        axes[0].set_xlabel(LBL[x_key])
        axes[0].set_ylabel(LBL[col_y])
        axes[0].set_title(LBL["galaxies"])
        fig.colorbar(h0[3], ax=axes[0], label=LBL["counts"])

        if weight_valid.any():
            h1 = axes[1].hist2d(
                x[weight_valid], y[weight_valid], bins=50, range=[CMD_X_RANGE, CMD_Y_RANGE],
                weights=w[weight_valid], cmap="viridis",
            )
            axes[1].set_xlabel(LBL[x_key])
            axes[1].set_ylabel(LBL[col_y])
            axes[1].set_title(LBL["sum_pmem"])
            fig.colorbar(h1[3], ax=axes[1], label=LBL["sum_pmem"])
        else:
            axes[1].hist2d(
                x[valid], y[valid], bins=50, range=[CMD_X_RANGE, CMD_Y_RANGE], cmap="viridis",
            )
            axes[1].set_xlabel(LBL[x_key])
            axes[1].set_ylabel(LBL[col_y])
            axes[1].set_title("Membership weights unavailable")

        cg_plot = _cg_xy_mask(bcg_mask, x, y)
        if cg_plot.any():
            for ax in axes:
                _plot_cg(ax, x[cg_plot], y[cg_plot], label=LBL["cg"])
            axes[0].legend()
    plt.show()

## Color-magnitude diagrams by color index

$m_z$ versus $g-r$, $r-i$, and $i-z$ for all members. Point color encodes $P_{\mathrm{mem}}$. The highlighted panel shows the color index recommended for this cluster's $z_{\mathrm{phot}}$. The central galaxy (CG) is marked with a red dashed circle when identified.

$$g-r = m_g - m_r,\quad r-i = m_r - m_i,\quad i-z = m_i - m_z$$

In [ ]:
if not members.empty and {"g_r", "r_i", "i_z", REF_MAG}.issubset(members.columns):
    cmd_specs = [
        ("g_r", "g-r", r"$z_{\mathrm{phot}} < 0.35$"),
                ("r_i", "r-i", r"$0.35 \leq z_{\mathrm{phot}} < 0.75$"),
        ("i_z", "i-z", r"$z_{\mathrm{phot}} \geq 0.75$"),
    ]
    fig, axes = _subplots_grid(1, 3, height=4.8, sharex=True)

    for ax, (col, label, note) in zip(axes, cmd_specs):
        xv = members[REF_MAG]
        yv = members[col]
        valid = xv.notna() & yv.notna()
        sc = ax.scatter(
            xv[valid], yv[valid],
            c=members.loc[valid, "pmem"] if "pmem" in members.columns else "C0",
            s=PMEM_SCATTER_SIZE, alpha=0.65, cmap="plasma", vmin=0, vmax=1, linewidths=0,
        )
        if col_y == col:
            ax.set_facecolor("#f5f5f5")
        cg_plot = _cg_xy_mask(bcg_mask, xv, yv)
        if cg_plot.any():
            _plot_cg(ax, xv[cg_plot], yv[cg_plot])
        ax.set_xlabel(LBL[REF_MAG])
        ax.set_ylabel(LBL[col])
        ax.set_title(f"{LBL[label]}  ({note})")
    cbar = fig.colorbar(sc, ax=axes, orientation="horizontal", pad=0.08, aspect=50)
    cbar.set_label(LBL["pmem"])
    plt.show()

## Scatter diagram and membership probability

**Left:** Recommended color versus $m_z$ at the cluster photometric redshift; point color encodes $P_{\mathrm{mem}}$. The CG is marked when identified.

**Right:** Distribution of membership probabilities among members (Castignani & Benoist 2016). The CG value is shown as a red dashed line when identified.

In [ ]:
if not members.empty and "pmem" in members.columns:
    fig, axes = _subplots_grid(1, 2, height=5.0)

    x = members[x_key]
    y = members[col_y]
    valid = x.notna() & y.notna() & members["pmem"].notna()

    sc = axes[0].scatter(
        x[valid], y[valid], c=members.loc[valid, "pmem"],
        s=PMEM_SCATTER_SIZE + 4, alpha=0.7, cmap="plasma", vmin=0, vmax=1, linewidths=0,
    )
    cg_plot = _cg_xy_mask(bcg_mask, x, y)
    if cg_plot.any():
        _plot_cg(axes[0], x[cg_plot], y[cg_plot], label=LBL["cg"])
    axes[0].set_xlabel(LBL[x_key])
    axes[0].set_ylabel(LBL[col_y])
    axes[0].set_title("Color-magnitude scatter")
    fig.colorbar(sc, ax=axes[0], label=LBL["pmem"])
    if cg_plot.any():
        axes[0].legend(fontsize=8)

    axes[1].hist(members["pmem"].dropna(), bins=40, color="steelblue", alpha=0.85, label="Members")
    if bcg_mask.any() and members.loc[bcg_mask, "pmem"].notna().any():
        axes[1].axvline(
            members.loc[bcg_mask, "pmem"].dropna().iloc[0],
            color="red", ls="--", lw=0.8, label=LBL["cg"],
        )
    axes[1].set_xlabel(LBL["pmem"])
    axes[1].set_ylabel(LBL["counts"])
    axes[1].set_title("$P_{\mathrm{mem}}$ distribution")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)

    plt.show()

## Magnitude histograms

Number counts (blue) and $P_{\mathrm{mem}}$-weighted distributions (orange) in each filter. The reference band $m_z$ is indicated (Benoist et al. 2025). Magnitude range: 15–24.

In [ ]:
if not members.empty:
    mag_specs = [(c, LBL[c]) for c in ("mag_g", "mag_r", "mag_i", "mag_z", "mag_y") if c in members.columns]
    ncols = min(len(mag_specs), 3)
    nrows = int(np.ceil(len(mag_specs) / ncols))
    fig, axes = _subplots_grid(nrows, ncols, height=3.4 * nrows)
    axes = np.atleast_1d(axes).ravel()

    for ax, (col, label) in zip(axes, mag_specs):
        vals = members[col].dropna()
        weights = members.loc[vals.index, "pmem"] if "pmem" in members.columns else None
        lw = 2.5 if col == REF_MAG else 2
        ax.hist(vals, bins=28, range=(15, 24), histtype="step", lw=lw, color="steelblue", label=LBL["counts"])
        if weights is not None:
            ax.hist(vals, bins=28, range=(15, 24), histtype="step", lw=1.5,
                    weights=weights, color="darkorange", label=LBL["weighted"])
        ax.set_xlabel(label)
        ax.set_ylabel(LBL["counts"])
        ax.set_title(label + (r"  (ref.)" if col == REF_MAG else ""))
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
    for ax in axes[len(mag_specs):]:
        ax.set_visible(False)
    plt.show()

## Membership probability versus photometry

$P_{\mathrm{mem}}$ as a function of the recommended color index, $m_i$, and galaxy photometric redshift.

In [ ]:
if not members.empty and "pmem" in members.columns:
    fig, axes = _subplots_grid(1, 3, height=4.0)
    specs = []
    if col_y in members.columns:
        specs.append((col_y, LBL[col_y]))
    if "mag_i" in members.columns:
        specs.append(("mag_i", LBL["mag_i"]))
    if "zphot" in members.columns:
        specs.append(("zphot", LBL["zphot_galaxy"]))

    for ax, (col, xlab) in zip(axes, specs):
        valid = members[col].notna() & members["pmem"].notna()
        ax.scatter(
            members.loc[valid, col], members.loc[valid, "pmem"],
            s=10, alpha=0.45, color="steelblue", label="Members",
        )
        cg_plot = _cg_xy_mask(bcg_mask, members[col], members["pmem"])
        if cg_plot.any():
            _plot_cg(ax, members.loc[cg_plot, col], members.loc[cg_plot, "pmem"], label=LBL["cg"])
        ax.set_xlabel(xlab)
        ax.set_ylabel(LBL["pmem"])
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    for ax in axes[len(specs):]:
        ax.set_visible(False)
    plt.show()

## Spatial distribution

**Left:** Angular offsets from the cluster center (arcsec), color-coded by $P_{\mathrm{mem}}$. Gray crosshairs mark the cluster center; the gray dashed circle shows $R_{\mathrm{amin}}$. The CG is marked when identified.

$$\Delta\mathrm{RA} = (\mathrm{RA}_i - \mathrm{RA}_{\mathrm{cl}})\,\cos(\mathrm{Dec}_{\mathrm{cl}})\,\times\,3600,\qquad \Delta\mathrm{Dec} = (\mathrm{Dec}_i - \mathrm{Dec}_{\mathrm{cl}})\,\times\,3600$$

**Right:** Same offsets on the sky with RA and Dec tick labels (1:1 aspect ratio). Gray crosshairs mark the cluster center; the gray dashed circle shows $R_{\mathrm{amin}}$. The CG is marked when identified.

In [ ]:
if (
    not members.empty
    and "dra_arcsec" in members.columns
    and "ddec_arcsec" in members.columns
    and np.isfinite(cluster_ra)
    and np.isfinite(cluster_dec)
):
    valid = _finite_xy_mask(members["dra_arcsec"], members["ddec_arcsec"])
    if valid.any():
        fig, axes = _subplots_grid(1, 2, height=7.0)
        dra = members.loc[valid, "dra_arcsec"]
        ddec = members.loc[valid, "ddec_arcsec"]
        pmem_vals = members.loc[valid, "pmem"] if "pmem" in members.columns else None
        use_pmem = pmem_vals is not None and pmem_vals.notna().any()

        sc = axes[0].scatter(
            dra, ddec,
            c=pmem_vals if use_pmem else "C0",
            s=PMEM_SCATTER_SIZE, alpha=0.65, cmap="plasma", vmin=0, vmax=1, linewidths=0,
        )
        axes[0].axhline(0, color="gray", ls=":", lw=1)
        axes[0].axvline(0, color="gray", ls=":", lw=1)
        cg_sp = _cg_xy_mask(bcg_mask, members["dra_arcsec"], members["ddec_arcsec"])
        cg_has_spatial = False
        if cg_sp.any():
            _plot_cg(axes[0], members.loc[cg_sp, "dra_arcsec"], members.loc[cg_sp, "ddec_arcsec"], label=LBL["cg"])
            cg_has_spatial = True
        elif _filled(_get(cluster, "ra_cg")) and _filled(_get(cluster, "dec_cg")):
            cg_ra = _flt(_get(cluster, "ra_cg"))
            cg_dec = _flt(_get(cluster, "dec_cg"))
            if np.isfinite(cg_ra) and np.isfinite(cg_dec):
                dra_cg = (cg_ra - cluster_ra) * np.cos(np.radians(cluster_dec)) * 3600
                ddec_cg = (cg_dec - cluster_dec) * 3600
                _plot_cg(axes[0], [dra_cg], [ddec_cg], label=LBL["cg"])
                cg_has_spatial = True
        _add_offset_radius_circle(axes[0], radius_arcsec)
        _symmetric_arcsec_limits(axes[0], dra, ddec, radius_arcsec)
        _set_equal_sky_axes(axes[0])
        axes[0].set_xlabel(LBL["dra"])
        axes[0].set_ylabel(LBL["ddec"])
        axes[0].set_title(LBL["offsets"])
        if cg_has_spatial or np.isfinite(radius_arcsec):
            axes[0].legend(fontsize=8)

        if "ra" in members.columns and "dec" in members.columns:
            axes[1].scatter(
                dra, ddec,
                s=PMEM_SCATTER_SIZE, alpha=0.5,
                c=pmem_vals if use_pmem else "C0",
                cmap="plasma", vmin=0, vmax=1, linewidths=0,
            )
            axes[1].axhline(0, color="gray", ls=":", lw=1)
            axes[1].axvline(0, color="gray", ls=":", lw=1)
            if cg_sp.any():
                _plot_cg(
                    axes[1],
                    members.loc[cg_sp, "dra_arcsec"],
                    members.loc[cg_sp, "ddec_arcsec"],
                    label=LBL["central_galaxy"],
                )
            elif cg_has_spatial:
                _plot_cg(axes[1], [dra_cg], [ddec_cg], label=LBL["central_galaxy"])
            _add_offset_radius_circle(axes[1], radius_arcsec)
            _symmetric_arcsec_limits(axes[1], dra, ddec, radius_arcsec)
            _set_equal_sky_axes(axes[1])
            _format_ra_dec_ticks(axes[1], cluster_ra, cluster_dec)
            axes[1].set_title(LBL["sky"])
            axes[1].legend(fontsize=8)

        if use_pmem:
            cbar = fig.colorbar(sc, ax=axes, orientation="horizontal", pad=0.08, aspect=50)
            cbar.set_label(LBL["pmem"])
        plt.show()